# MCP Lab 1 — Building Your First MCP Server (Solutions)

The **Model Context Protocol** (MCP) is an open protocol for connecting LLM
applications to external tools, data sources, and prompt libraries. Instead of every
application reinventing tool integration, an MCP server exposes capabilities once and
any MCP-aware client can use them — Claude Desktop, IDE plugins, custom agents.

In this lab you build an MCP server from scratch.

## Learning objectives

By the end of this lab you will be able to:

1. Explain what MCP is and what problem it solves.
2. Distinguish the three MCP primitives — **tools, resources, prompts** — and pick the
   right one for a given capability.
3. Build a server with the official `mcp` Python SDK (`FastMCP`) that wraps a SQLite
   database.
4. Launch the server as a stdio subprocess and inspect its capabilities from a client.
5. Call tools, read resources, and fetch prompts programmatically.

## Prerequisites

- `pip install -r requirements.txt`
- `shop.db` exists at `../talk_to_your_data/shop.db` (run `python ../talk_to_your_data/seed_db.py` if not).
- Python 3.10+.

## 1. What is MCP?

Most LLM applications today have the same problem: the model needs to call out to the
world — read a database, hit an internal API, look at a file. The model can do that
only through **tool calling**, which means somebody has to:

- define the tool's JSON schema,
- implement the tool,
- register it with the LLM client,
- wire authentication, error handling, observability for it,
- do this *in every application* that wants to use it.

MCP factors this out. A server exposes tools (and resources, and prompts) over a
standard protocol. Any MCP-aware client — Claude Desktop, an IDE, your custom
agent — can connect and use them with no per-application code.

Two transports:

- **stdio** — the server is a subprocess of the client; they speak JSON-RPC over its
  stdin/stdout. Simplest, used for local servers. This is what we use in the lab.
- **HTTP / Server-Sent Events** — the server is a long-running HTTP service. Used for
  shared, remote servers.

Either way, the protocol is the same: a JSON-RPC handshake, then `list_*` and `call_*`
messages.

## 2. The three primitives

| Primitive | Who triggers it       | What it returns                          | Example                                  |
|-----------|-----------------------|------------------------------------------|------------------------------------------|
| **Tool**     | The model (function call) | A result of executing some action      | "search the database", "send an email"   |
| **Resource** | The application/user (read) | Read-only data — text or binary       | "the schema file", "today's metrics CSV" |
| **Prompt**   | The user (selected)         | A pre-authored message template       | "summarize the last 7 days of orders"    |

- **Tools** are the most familiar piece — they are the same idea as OpenAI/Anthropic
  tool calling, just standardized.
- **Resources** are data the model can *read* without taking an action. Think of a
  file the application surfaces into the prompt context.
- **Prompts** are templates the user (not the model) can invoke. The server defines
  them; the client surfaces them in its UI (e.g. as slash commands).

## 3. Setup

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import asyncio
import sys
from pathlib import Path

# Defensive: on some Windows + Jupyter combos the default event loop policy can't
# spawn subprocesses; the proactor policy can. This is harmless if already set.
if sys.platform == "win32":
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except Exception:
        pass

import nest_asyncio
nest_asyncio.apply()

DB_PATH = (Path.cwd().parent / "talk_to_your_data" / "shop.db").resolve()
assert DB_PATH.exists(), f"shop.db not found at {DB_PATH}. Run seed_db.py first."
print("shop.db ok:", DB_PATH)

> **A note on async in Jupyter.** MCP's stdio transport uses subprocesses, which means we need
> `asyncio`. Jupyter (IPython 7+) supports top-level `await`, so most cells just call
> `await ...` directly. If your environment uses a different event-loop policy, the setup cell
> applies `nest_asyncio` defensively, which is harmless when not needed.

## 4. A minimum viable server

Let's build the smallest server that works — one tool that adds two numbers. The
`FastMCP` class uses Python decorators to register handlers: `@mcp.tool()`,
`@mcp.resource(uri)`, `@mcp.prompt()`. The function signature and docstring are
introspected to build the JSON schema the model sees.

In [ ]:
%%writefile hello_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("hello")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


if __name__ == "__main__":
    mcp.run()

### Talking to the server from this notebook

The server, when run, takes over the process and speaks the MCP protocol on its stdio.
To talk to it from Python we **launch it as a subprocess** with
`StdioServerParameters`, then create a `ClientSession` over its streams.

We'll build a real client in Lab 2. For now this minimal block is enough to confirm
the server works.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

hello_params = StdioServerParameters(command=sys.executable, args=["hello_server.py"])

async with stdio_client(hello_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        print("tools:", [t.name for t in tools.tools])

        result = await session.call_tool("add", {"a": 2, "b": 40})
        # Tool results are a list of content items (TextContent, ImageContent, …).
        # We extract the text fields.
        text = "".join(getattr(c, "text", "") for c in result.content)
        print("add(2, 40) =", text)

## 5. Designing the shop server

Now we'll build a more useful server: one that lets a model query the `shop.db`
database from the Text-to-SQL labs. This is *intentional* — the same database is now
accessible two ways:

- Through the **text-to-SQL pipeline** (LLM generates SQL).
- Through a set of **typed MCP tools** (LLM calls specific, audited functions).

The MCP tool path is safer: every tool is hand-written, with explicit parameters and
validation. The trade-off is breadth — you can only answer what your tools cover.

### What we'll expose

**Tools** (the model can invoke):

- `list_products(category: Optional[str])`
- `top_products_by_rating(limit: int = 5)`
- `get_customer_orders(customer_id: int)`
- `search_reviews(product_name: str)`

**Resources** (the application can read):

- `schema://shop` — the database DDL
- `data://categories` — the list of product categories

**Prompts** (the user can select):

- `summarize_orders(customer_id: int)` — friendly per-customer summary

## 6. Implementing the tools

The next cell uses Jupyter's `%%writefile` magic to write `shop_server.py` from this
cell's contents. Fill in the TODOs inside the cell, then run it: Jupyter saves the
file, and Section 9 below launches a fresh server process from it.

Tips for the SQL inside the tools:

- Use the `_query(sql, params)` helper. It opens a read-only connection and returns a
  list of dicts.
- Pass user input as **parameters**, never via f-strings.

## 7. Implementing the resources

## 8. Implementing the prompts

In [ ]:
# %%writefile shop_server.py
"""A small MCP server that exposes the shop.db database from the TTYD labs.

Run directly to talk on stdio:
    python shop_server.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

from mcp.server.fastmcp import FastMCP

# The shop.db database lives one folder up (in talk_to_your_data/).
DB_PATH = Path(__file__).resolve().parent.parent / "talk_to_your_data" / "shop.db"

mcp = FastMCP("shop")


def _query(sql: str, params: tuple = ()) -> list[dict]:
    uri = f"file:{DB_PATH.as_posix()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, params).fetchall()]


# --------------------------- Tools ---------------------------

@mcp.tool()
def list_products(category: str | None = None) -> list[dict]:
    """List products in the catalogue, optionally filtered by category."""
    if category:
        return _query(
            "SELECT product_id, name, category, price, stock FROM products "
            "WHERE category = ? ORDER BY name",
            (category,),
        )
    return _query("SELECT product_id, name, category, price, stock FROM products ORDER BY name")


@mcp.tool()
def top_products_by_rating(limit: int = 5) -> list[dict]:
    """Return the products with the highest average review rating."""
    return _query(
        "SELECT p.product_id, p.name, p.category, "
        "       ROUND(AVG(r.rating), 2) AS avg_rating, COUNT(r.review_id) AS n_reviews "
        "FROM products p JOIN reviews r ON r.product_id = p.product_id "
        "GROUP BY p.product_id "
        "HAVING n_reviews >= 3 "
        "ORDER BY avg_rating DESC, n_reviews DESC "
        "LIMIT ?",
        (int(limit),),
    )


@mcp.tool()
def get_customer_orders(customer_id: int) -> list[dict]:
    """Get all orders for a specific customer, most recent first."""
    return _query(
        "SELECT order_id, order_date, status, total_amount FROM orders "
        "WHERE customer_id = ? ORDER BY order_date DESC",
        (int(customer_id),),
    )


@mcp.tool()
def search_reviews(product_name: str) -> list[dict]:
    """Return all reviews whose product name contains the given substring (case-insensitive)."""
    return _query(
        "SELECT p.name AS product, r.rating, r.comment, r.review_date "
        "FROM reviews r JOIN products p ON p.product_id = r.product_id "
        "WHERE LOWER(p.name) LIKE LOWER(?) "
        "ORDER BY r.review_date DESC",
        (f"%{product_name}%",),
    )


# ------------------------- Resources -------------------------

@mcp.resource("schema://shop")
def shop_schema() -> str:
    """The full CREATE TABLE DDL for the shop database."""
    rows = _query(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    )
    return "\n\n".join(r["sql"] for r in rows if r["sql"])


@mcp.resource("data://categories")
def categories() -> str:
    """A newline-separated list of distinct product categories."""
    rows = _query("SELECT DISTINCT category FROM products ORDER BY category")
    return "\n".join(r["category"] for r in rows)


# -------------------------- Prompts --------------------------

@mcp.prompt()
def summarize_orders(customer_id: int) -> str:
    """A prompt template asking for a friendly summary of a customer's orders."""
    return (
        f"You are a customer-support assistant. Use the available tools to fetch the "
        f"orders for customer #{customer_id}, then write a short, friendly summary: "
        f"how many orders they have placed, the total amount they have spent, and the "
        f"status breakdown. Be concise."
    )


if __name__ == "__main__":
    mcp.run()

## 9. Inspecting your server

Now that `shop_server.py` exists on disk, we launch it and ask: what tools, resources,
and prompts does it advertise? This is exactly what an MCP-aware client (Claude
Desktop, an IDE plugin, etc.) would do on connection.

In [ ]:
shop_params = StdioServerParameters(command=sys.executable, args=["shop_server.py"])

async with stdio_client(shop_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        print("=== tools ===")
        for t in tools.tools:
            print(f"  {t.name}: {t.description}")

        resources = await session.list_resources()
        print("\n=== resources ===")
        for r in resources.resources:
            print(f"  {r.uri}: {r.description}")

        prompts = await session.list_prompts()
        print("\n=== prompts ===")
        for p in prompts.prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"  {p.name}({args}): {p.description}")

### Calling a tool

In [ ]:
async with stdio_client(shop_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.call_tool("top_products_by_rating", {"limit": 3})
        for c in result.content:
            if hasattr(c, "text"):
                print(c.text)

### Reading a resource

In [ ]:
async with stdio_client(shop_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        res = await session.read_resource("schema://shop")
        for c in res.contents:
            if hasattr(c, "text"):
                print(c.text)

### Fetching a prompt

In [ ]:
async with stdio_client(shop_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        prm = await session.get_prompt("summarize_orders", {"customer_id": "1"})
        for msg in prm.messages:
            print(f"[{msg.role}] {msg.content.text}")

## Summary

You built an MCP server that exposes:

- Four **tools** wrapping common shop queries.
- Two **resources** for read-only schema/data.
- One **prompt** for a pre-baked customer-summary workflow.

And you watched a client introspect the server and use each primitive.

In **Lab 2** we replace the throw-away client cells above with a real client and wire
it to an OpenAI tool-calling agent — so a natural-language question goes through the
agent → MCP → SQLite → back to a friendly answer.

## Exercises

1. **A new tool.** Add `category_revenue(year: int)` that returns total revenue per
   product category for a given year. Decide which existing queries to compose.
2. **A new resource.** Add `stats://shop` that returns a one-paragraph natural-
   language summary of the database (row counts, date range covered, number of
   categories). Resources are computed on demand, so this can be live.
3. **A safer SQL tool.** Add a tool `run_safe_select(sql: str)` that wires the
   `validate_sql_strict` function from TTYD Lab 2 in front of an `execute_safely`
   call. Decide what to return when validation fails: a thrown error, or a structured
   `{ok: false, reason: ...}`?
4. **Authentication-shaped problem.** Imagine adding `get_customer_orders` as a tool a
   *customer* could call. How would you carry the customer's identity to the server
   when the client launches it as a subprocess? (Sketch — don't implement.)